# Manderson & Considine (2025)

Illinois county-level LOCA2 indicators. Keep this notebook and `loca2_common.py` in the same folder. Open Jupyter in that folder and run the cells from top to bottom. Each notebook uses its own kernel.

Existing files are skipped (`OVERWRITE = False`). County aggregation uses cosine-latitude area correction.

This workflow requires both relative-humidity inputs as well as temperature and precipitation.

Go to loca2_common.py to set your input and output path. 

Indicators calculated: Daily mean temperature within specified bins (degF), Monthly total precipitation (inches) and Monthly Average Relative Humidity (%)

## 1. Import shared functions

Required packages: numpy, xarray, dask, netCDF4, cftime, pandas, geopandas, regionmask, and tqdm. Use your existing Keeling environment.

In [1]:
from __future__ import annotations

import gc
import glob
import re
from pathlib import Path
import numpy as np
import xarray as xr
import geopandas as gpd
from tqdm.auto import tqdm
from pathlib import Path
import numpy as np
import xarray as xr
import loca2_common as common
from loca2_common import (
    aggregate_to_illinois_counties,
    daily_mean_temperature,
    monthly_mean,
    monthly_sum,
    monthly_temperature_bin_counts,
    to_fahrenheit,
)

## 2. Paper settings

In [3]:
COUNTY_SHAPEFILE = common.DATA_ROOT / 'Illinois_County_Boundaries' / 'IL_BNDY_County_Py.shp'
counties = gpd.read_file(COUNTY_SHAPEFILE)
counties = counties.to_crs('EPSG:4326').reset_index(drop=True)
counties['county_id'] = '17' + counties['CO_FIPS'].astype(int).astype(str).str.zfill(3) #Illinois State code is 17. CO_FIPS is a 5 digit county code that starts with the state code. 
counties['county_name'] = counties['COUNTY_NAM'].astype(str).str.strip()
PAPER_NAME = 'Manderson & Considine (2025)'
REQUIRED_VARIABLES = ('tasmin', 'tasmax', 'pr', 'hursmin', 'hursmax')
OVERWRITE = True

## 3. Check paths and available runs

This cell lists the workload and loads county boundaries. It does not write climate output.

In [4]:
output_directory = common.PAPER_OUTPUT_ROOT / common.paper_directory_name(PAPER_NAME)
jobs = common.discover_paper_runs(REQUIRED_VARIABLES)
existing = sum(
    (output_directory / (
        f"{common.safe_token(PAPER_NAME)}_"
        f"{common.safe_token(model)}_{common.safe_token(member)}_{scenario}.nc"
    )).exists()
    for model, member, scenario in jobs
)
print("Input root:", common.PAPER_INPUT_ROOT)
print("Output folder:", output_directory)
print("Eligible runs:", len(jobs))
print("Existing files to skip:", existing if not OVERWRITE else 0)
print("Runs to process:", len(jobs) - existing if not OVERWRITE else len(jobs))
if not jobs:
    raise FileNotFoundError("No eligible runs found. Check input paths and required variables.")
counties = common.load_counties()
print("Counties:", len(counties))

Input root: /data/keeling/a/tahsina2/b/Climate_Impact/Data/LOCA2_Illinois
Output folder: /data/keeling/a/tahsina2/b/Climate_Impact/Data/LOCA2_Illinois/temp/Manderson & Considine (2025)
Eligible runs: 125
Existing files to skip: 0
Runs to process: 125
Counties: 102


## 4. Paper indicator calculation

In [22]:
def calculate_manderson_considine(inputs):
    """
    Calculate monthly Manderson and Considine indicators and
    aggregate the results to Illinois counties.
    """
    tmean_c = daily_mean_temperature(inputs['tasmin'], inputs['tasmax'])
    tmean_f = to_fahrenheit(tmean_c)
    edges_f = [-np.inf, 15.0, 25.0, 35.0, 45.0, 55.0, 65.0, 75.0, 85.0, np.inf]
    labels = ['tmean_days_lt15F', 'tmean_days_15_25F', 'tmean_days_25_35F', 'tmean_days_35_45F', 'tmean_days_45_55F', 'tmean_days_55_65F', 'tmean_days_65_75F', 'tmean_days_75_85F', 'tmean_days_ge85F']
    variables = monthly_temperature_bin_counts(temperature=tmean_f, edges=edges_f, labels=labels)
    variables['monthly_precipitation'] = monthly_sum(inputs['pr'])
    rh_min, rh_max = xr.align(inputs['hursmin'], inputs['hursmax'], join='exact')
    for name, data in {'hursmin': rh_min, 'hursmax': rh_max}.items():
        units = str(data.attrs.get('units', '')).strip().lower()
        if units not in {'%', 'percent', 'percentage'}:
            raise ValueError(f"{name} must be in percent, but its units are {data.attrs.get('units')!r}.")
    daily_mean_rh = (0.5 * (rh_min + rh_max)).clip(min=0.0, max=100.0).rename('daily_mean_relative_humidity')
    daily_mean_rh.attrs.update(units='percent', long_name='daily mean relative humidity calculated as (daily minimum + daily maximum) / 2')
    variables['monthly_mean_relative_humidity'] = monthly_mean(daily_mean_rh)
    result = xr.Dataset(variables)
    for variable in labels:
        result[variable].attrs.update(units='days', frequency='monthly', temperature_statistic='daily mean temperature calculated as (tasmin + tasmax) / 2')
    result['monthly_precipitation'].attrs.update(units='inch', long_name='monthly total precipitation', temporal_aggregation='sum of daily precipitation')
    result['monthly_mean_relative_humidity'].attrs.update(units='percent', long_name='monthly mean relative humidity', daily_calculation='(rel_humid_min + rel_humid_max) / 2', temporal_aggregation='mean of daily mean relative humidity')
    result.attrs.update(reference_temperature_bin='55 to less than 65 degrees Fahrenheit', relative_humidity_file_directories='hursmin and hursmax', relative_humidity_internal_variables='rel_humid_min and rel_humid_max', relative_humidity_statistic='monthly mean of daily average relative humidity')
    result = aggregate_to_illinois_counties(dataset=result, counties=counties)
    return result

## 5. Generate output

Existing matching filenames are skipped. A skipped file is not recalculated or checked for completeness.

In [23]:
common.run_paper(
    paper_name=PAPER_NAME,
    required_variables=REQUIRED_VARIABLES,
    calculation_function=calculate_manderson_considine,
    overwrite=OVERWRITE,
)

Manderson & Considine (2025):   0%|          | 0/125 [00:00<?, ?run/s]

WROTE: /data/keeling/a/tahsina2/b/Climate_Impact/Data/LOCA2_Illinois/Manderson & Considine (2025)/Manderson_Considine_2025__ACCESS-CM2_r1i1p1f1_ssp245.nc


KeyboardInterrupt: 

## 6. Inspect an output file

In [ ]:
files = sorted(output_directory.glob(f"{common.safe_token(PAPER_NAME)}_*.nc"))
if files:
    with xr.open_dataset(files[0], decode_timedelta=False) as dataset:
        print("Example:", files[0].name)
        print("Dimensions:", dict(dataset.sizes))
        print("Variables:", list(dataset.data_vars))
        print("Spatial resolution:", dataset.attrs.get("spatial_resolution"))
        assert dataset.sizes.get("county") == 102, "Output is not a 102-county dataset."
else:
    print("No output files found. Review the processing messages above.")